# Parda — web app M2, step A: GLiNER golden cases for the browser port

Runs the **shipped** GLiNER (ONNX, int8_embed, 555 MB) on 160 real OCR text chunks (scans + phone photos, Tesseract +
EasyOCR, 3 languages) and records exactly what the network receives, what it returns and which spans GLiNER keeps.
The browser version must reproduce all of it. Also saves the tokenizer and GLiNER's source code.

**Settings:** Accelerator **None (CPU)** · Internet **On** · **Input:** Your Work → the **08b** notebook (saved OCR)
**Secrets:** `GITHUB_TOKEN`, `HF_TOKEN`. Run all cells (~15 min), then download **`gliner_goldens.zip`** from Output.


In [ ]:
# 1. Code + secrets
import os, subprocess
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
GH = secrets.get_secret("GITHUB_TOKEN")
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
REPO, CODE = "chetanjagan/Parda", "/tmp/Parda"
if not os.path.exists(CODE):
    subprocess.run(["git", "clone", "-q", f"https://{GH}@github.com/{REPO}.git", CODE], check=True)
    subprocess.run(["git", "-C", CODE, "remote", "set-url", "origin", f"https://github.com/{REPO}.git"], check=True)
else:
    subprocess.run(["git", "-C", CODE, "pull", "-q", f"https://{GH}@github.com/{REPO}.git", "main"], check=True)
os.chdir(CODE)
print(subprocess.run(["git", "log", "--oneline", "-3"], capture_output=True, text=True).stdout)


In [ ]:
# 2. GLiNER + ONNX runtime
!pip install -q gliner onnx onnxruntime
!python -c "import gliner, onnxruntime; print('gliner', gliner.__version__, '| onnxruntime', onnxruntime.__version__)"


In [ ]:
# 3. Saved OCR (from 08b) + the shipped model (from your HF repo parda-onnx-v1)
import json, shutil
from huggingface_hub import HfApi, snapshot_download
from parda.fsutil import find_path
IN, W = "/kaggle/input", "/kaggle/working"
first = lambda *rels: next((h[0] for r in rels for h in [find_path(IN, r)] if h), None)
C_SEEN = os.path.dirname(first("cache_seen/ocr_easyocr.jsonl") or "")
C_OOD = os.path.dirname(first("cache_ood/ocr_easyocr.jsonl") or "")
assert C_SEEN and C_OOD, "attach the 08b notebook (cache_seen, cache_ood)"
HF_USER = HfApi(token=os.environ["HF_TOKEN"]).whoami()["name"]
snapshot_download(f"{HF_USER}/parda-onnx-v1", local_dir=f"{W}/onnx", token=os.environ["HF_TOKEN"],
                  allow_patterns=["gliner/model_int8_embed.onnx", "gliner/parda_onnx.json"])
meta = json.load(open(f"{W}/onnx/gliner/parda_onnx.json"))
print("OCR:", C_SEEN, "|", C_OOD, "| model default:", meta.get("default"), "| files:", meta["files"])
assert meta.get("default") == "int8_embed", "expected the int8_embed model chosen in notebook 10"


In [ ]:
# 4. Record the golden cases (~10 min on CPU)
OCR = " ".join(f"{d}/{f}" for d in (C_SEEN, C_OOD) for f in ("ocr_tesseract.jsonl", "ocr_easyocr.jsonl"))
!python tools/make_gliner_goldens.py --model {W}/onnx/gliner --ocr {OCR} --n 160 --out {W}/gliner_goldens 2>&1 | grep -v -i warning | tail -30


In [ ]:
# 5. One zip to download (Output tab -> gliner_goldens.zip), and a quick look
g = json.load(open(f"{W}/gliner_goldens/gliner_goldens.json"))
c = next(c for c in g["cases"] if c["entities"])
print("example chunk:", repr(c["chunk"][:150]))
print("words:", c["words"][:12])
print("inputs:", {k: v["shape"] for k, v in c["inputs"].items()})
print("entities:", c["entities"][:4])
shutil.make_archive(f"{W}/gliner_goldens", "zip", f"{W}/gliner_goldens")
print(f"gliner_goldens.zip: {os.path.getsize(f'{W}/gliner_goldens.zip') / 2**20:.1f} MB")
